In [1]:
import json
import os
import sys
from typing import Optional

sys.path.insert(0, "../")
from utils import get_args, MyProgressBar
from cancerfoundation.model.model import CancerFoundation
from cancerfoundation.data.data_module import SingleCellDataModule
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
	
import copy
import gc
import json
import os
from pathlib import Path
import sys
import time
import traceback
from typing import List, Tuple, Dict, Union, Optional
import warnings

import torch
from anndata import AnnData
import scanpy as sc
import numpy as np
import wandb
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
import scanpy as sc
import torch
import numpy as np
from cancerfoundation.model.model import CancerFoundation
import json
from cancerfoundation.data.preprocess import binning
import pandas as pd

model_name = "my_init_4gpu_lrx2"

# Load model
with open(f"../save/{model_name}/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    f'../save/{model_name}/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)
model.eval()
model.cuda()

CancerFoundation(
  (criterion): MSE()
  (model): OptimizedModule(
    (_orig_mod): TransformerModule(
      (value_encoder): TheirContinuousValueEncoder(
        (dropout): Dropout(p=0.2, inplace=False)
        (linear1): Linear(in_features=1, out_features=256, bias=True)
        (activation): ReLU()
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (criterion_conditions): CrossEntropyLoss()
      (criterion): MSE()
      (condition_encoders): ModuleDict(
        (technology): ConditionEncoder(
          (embedding): Embedding(15, 256)
          (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
      )
      (transformer_encoder): CFGenerator(
        (layers): ModuleList(
          (0-5): 6 x CFLayer(
            (self_attn): MHA(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=256, o

In [4]:
model.model.transformer_encoder.layers[0].linear1.weight

Parameter containing:
tensor([[-0.0558, -0.0020,  0.0523,  ...,  0.0261, -0.0271, -0.0515],
        [-0.1216,  0.0608, -0.0940,  ...,  0.0248, -0.0858, -0.0349],
        [ 0.0341,  0.0327, -0.0037,  ...,  0.0866,  0.0232, -0.0051],
        ...,
        [-0.0306, -0.0179,  0.0061,  ..., -0.0086,  0.0060, -0.0320],
        [-0.0198,  0.0723, -0.1076,  ...,  0.0006,  0.1043, -0.0796],
        [-0.0646, -0.0595, -0.0353,  ...,  0.0644, -0.0847, -0.0197]],
       device='cuda:0', requires_grad=True)

In [5]:
def reset_all_weights(model):
    """Recursively reset all parameters in the model"""
    @torch.no_grad()
    def init_weights(m):
        if hasattr(m, 'reset_parameters'):
            m.reset_parameters()
    
    model.apply(init_weights)

In [6]:
reset_all_weights(model)

In [7]:
model.model.transformer_encoder.layers[0].linear1.weight

Parameter containing:
tensor([[-0.0062, -0.0291,  0.0201,  ..., -0.0320,  0.0388, -0.0598],
        [-0.0490,  0.0144,  0.0283,  ..., -0.0432,  0.0339, -0.0370],
        [-0.0212,  0.0270,  0.0139,  ..., -0.0443,  0.0002, -0.0130],
        ...,
        [ 0.0513, -0.0390,  0.0412,  ..., -0.0256,  0.0269,  0.0232],
        [ 0.0551,  0.0123,  0.0587,  ...,  0.0312, -0.0311, -0.0622],
        [ 0.0252, -0.0163, -0.0560,  ..., -0.0590, -0.0557,  0.0559]],
       device='cuda:0', requires_grad=True)

In [9]:
adata = sc.read_h5ad("./neftel_ss2/ground_truth/neftel_ss2.h5ad")

In [ ]:
model = CancerFoundation(
    n_bins=args.n_bins,
    vocab=datamodule.vocab,
    input_emb_style=args.input_emb_style,
    max_seq_len=args.max_seq_len,
    input_style=args.input_style,
    mask_ratio=args.mask_ratio,
    TRUNC_BY_SAMPLE=args.trunc_by_sample,
    training_tasks=args.training_tasks,
    embsize=args.embsize,
    nheads=args.nheads,
    d_hid=args.d_hid,
    nlayers=args.nlayers,
    dropout=args.dropout,
    lr=args.lr,
    epochs=args.epochs,
    warmup_ratio_or_step=args.warmup_ratio_or_step,
    scheduler_interval=args.scheduler_interval,
    scheduler_factor=args.scheduler_factor,
    loss_type=args.loss,
    do_dat=args.do_dat,
    no_invert_dat=args.no_invert_dat,
    conditions=args.conditions,
    conditions_nums=datamodule.conditions_nums if args.conditions else None,
    mvc_decoder_style=args.mvc_decoder_style,
    scale_zero_expression=args.scale_zero_expression,
    data_path=args.train_path,
    do_mvc=args.do_mvc,
    zero_percentages=args.zero_percentages,
    balance_primary=args.balance_primary,
    balance_secondary=args.balance_secondary,
    compile_model=args.compile,
    activation=args.activation,
    norm_first=args.norm_first,
    cell_emb_style=args.cell_emb_style,
    batchnorm=args.batchnorm,
    explicit_zero_prob=args.explicit_zero_prob,
    dat_scale=args.dat_scale,
    normalise_bins=args.normalise_bins,
    where_condition=args.where_condition,
    gen_method=args.gen_method,
    their_init_weights=args.their_init_weights,
)